# nb_img_00 — Image Manifest and Stratified 5% Seed Split

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

## Section 1 — Build manifest CSV

In [2]:
import pandas as pd
from pathlib import Path
from utils.image_config import IMAGE_DATA_DIR, IMAGE_CLASS_NAMES, IMAGE_MANIFEST_PATH

rows = []
for class_name in IMAGE_CLASS_NAMES:
    label_idx = IMAGE_CLASS_NAMES.index(class_name)
    class_dir = IMAGE_DATA_DIR / class_name
    for img_path in sorted(class_dir.glob("*.jpeg")) + sorted(class_dir.glob("*.jpg")) + sorted(class_dir.glob("*.png")):
        rows.append({
            "filepath": str(img_path),
            "true_label": class_name,
            "label_idx": label_idx,
            "is_seed": False,
        })

manifest = pd.DataFrame(rows)
print(f"Total images: {len(manifest)}")
print(manifest["true_label"].value_counts())

Total images: 5500
true_label
butterfly    550
cat          550
chicken      550
cow          550
dog          550
elephant     550
horse        550
sheep        550
spider       550
squirrel     550
Name: count, dtype: int64


## Section 2 — Stratified 5% seed split

In [3]:
from sklearn.model_selection import train_test_split
from utils.image_config import IMAGE_LABEL_FRACTION, IMAGE_SEED

seed_idx, _ = train_test_split(
    manifest.index,
    test_size=1 - IMAGE_LABEL_FRACTION,
    stratify=manifest["label_idx"],
    random_state=IMAGE_SEED,
)
manifest.loc[seed_idx, "is_seed"] = True
manifest.to_csv(IMAGE_MANIFEST_PATH, index=False)
print(f"Seed images: {manifest['is_seed'].sum()} ({manifest['is_seed'].mean():.1%})")
print(manifest.groupby("true_label")["is_seed"].sum())

Seed images: 275 (5.0%)
true_label
butterfly    28
cat          28
chicken      27
cow          27
dog          27
elephant     28
horse        27
sheep        28
spider       28
squirrel     27
Name: is_seed, dtype: int64


## Section 3 — Verify

In [4]:
loaded = pd.read_csv(IMAGE_MANIFEST_PATH)
assert len(loaded) == 5500, f"Expected 5500 rows, got {len(loaded)}"
assert set(loaded["true_label"].unique()) == set(IMAGE_CLASS_NAMES)
assert loaded["is_seed"].sum() >= 270  # ~5% of 5500
assert loaded.groupby("true_label")["is_seed"].sum().min() >= 25
print("Manifest OK")

Manifest OK
